In [7]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import corner
import ultranest
from utils.parse_data import *

In [8]:
df = read_file("data/mass.txt")
df = extract_relevant_data(df)

print(df.head())

     A    Z    N  EL  binding_energy_keV  binding_energy_unc
0  1.0  0.0  1.0   n             0.00000             0.00000
1  1.0  1.0  0.0   H             0.00000             0.00000
2  2.0  1.0  1.0   H          1112.28310             0.00020
3  3.0  1.0  2.0   H          2827.26540             0.00030
4  3.0  2.0  1.0  He          2572.68044             0.00015


**Likelihood and systematic uncertainty**

We use a Gaussian likelihood with an extra systematic term: $$\sigma_{\mathrm{sys}}$$ so that:

$$\sigma_i^2 = \sigma_{i,\mathrm{exp}}^2 + \sigma_{\mathrm{sys}}^2.$$

The log-likelihood becomes:

$$\log L = -\tfrac{1}{2}\sum_i\left[\frac{(B_i^{\mathrm{obs}}-B_i^{\mathrm{mod}})^2}{\sigma_i^2} + \ln(2\pi\sigma_i^2)\right]$$

In [9]:
# Bethe-Weizsäcker model WITHOUT pairing term, with prior and likelihood (uses same sigma_sys treatment)
def bethe_weizsacker_per_nucleon_no_pairing(A, Z, a_v, a_s, a_c, a_sym):
    A = np.array(A, dtype=float)
    Z = np.array(Z, dtype=float)
    B = a_v*A - a_s*A**(2.0/3.0) - a_c*Z*(Z-1)/A**(1.0/3.0) - a_sym*(A-2*Z)**2/A
    return B/A  # Return binding energy per nucleon

# Prior transform for no-pairing model: 5 parameters (a_v,a_s,a_c,a_sym,sigma_sys)
def prior_transform_nopair(u):
    a_v = 5.0 + u[0]*35.0      # 5-40 MeV
    a_s = 0.0 + u[1]*50.0      # 0-50 MeV
    a_c = 0.0 + u[2]*1.5       # 0-1.5 MeV
    a_sym = 0.0 + u[3]*60.0    # 0-60 MeV
    log10_sigma = -3.0 + u[4]*4.0  # sigma_sys from 1e-3 to 10 (same units as B)
    sigma_sys = 10**log10_sigma
    return [a_v, a_s, a_c, a_sym, sigma_sys]

def log_likelihood_nopair(params, A, Z, B_obs, B_err):
    # params: a_v,a_s,a_c,a_sym,sigma_sys
    a_v,a_s,a_c,a_sym,sigma_sys = params
    B_model = bethe_weizsacker_per_nucleon_no_pairing(A, Z, a_v,a_s,a_c,a_sym)
    sigma2 = B_err**2 + sigma_sys**2
    ll = -0.5*np.sum((B_obs - B_model)**2/sigma2 + np.log(2*np.pi*sigma2))
    return ll

print('Defined no-pairing model, prior_transform_nopair and log_likelihood_nopair')

Defined no-pairing model, prior_transform_nopair and log_likelihood_nopair


In [10]:
# Bethe-Weizsäcker model WITH pairing term, with prior and likelihood (uses same sigma_sys treatment)
def bethe_weizsacker_per_nucleon_with_pairing(A, Z, a_v, a_s, a_c, a_sym, a_p):
    A = np.array(A, dtype=float)
    Z = np.array(Z, dtype=float)
    pairing_term = np.zeros_like(A)
    even_Z = (Z % 2 == 0)
    even_N = ((A - Z) % 2 == 0)
    pairing_term[even_Z & even_N] = a_p / A[even_Z & even_N]**0.5
    pairing_term[~even_Z & ~even_N] = -a_p / A[~even_Z & ~even_N]**0.5
    B = (a_v*A - a_s*A**(2.0/3.0) - a_c*Z*(Z-1)/A**(1.0/3.0) - a_sym*(A-2*Z)**2/A + pairing_term)
    return B/A # Return binding energy per nucleon

def prior_transform_with_pairing(u):
    a_v = 5.0 + u[0]*35.0      # 5-40 MeV
    a_s = 0.0 + u[1]*50.0      # 0-50 MeV
    a_c = 0.0 + u[2]*1.5       # 0-1.5 MeV
    a_sym = 0.0 + u[3]*60.0    # 0-60 MeV
    a_p = 0.0 + u[4]*30.0      # 0-30 MeV (pairing term)
    log10_sigma = -3.0 + u[5]*4.0  # sigma_sys from 1e-3 to 10 (same units as B)
    sigma_sys = 10**log10_sigma
    return [a_v, a_s, a_c, a_sym, a_p, sigma_sys]

def log_likelihood_with_pairing(params, A, Z, B_obs, B_err):
    # params: a_v,a_s,a_c,a_sym,a_p,sigma_sys
    a_v,a_s,a_c,a_sym,a_p,sigma_sys = params
    B_model = bethe_weizsacker_per_nucleon_with_pairing(A, Z, a_v,a_s,a_c,a_sym,a_p)
    sigma2 = B_err**2 + sigma_sys**2
    ll = -0.5*np.sum((B_obs - B_model)**2/sigma2 + np.log(2*np.pi*sigma2))
    return ll

In [11]:
# Dati osservati
A = df['A'].to_numpy()
Z = df['Z'].to_numpy()
B_obs = df['binding_energy_keV'].to_numpy()
B_err = df['binding_energy_unc'].to_numpy()

def loglike_nopair(theta):
    return log_likelihood_nopair(theta, A, Z, B_obs, B_err)

def loglike_with_pairing(theta):
    return log_likelihood_with_pairing(theta, A, Z, B_obs, B_err)

sampler_nopair = ultranest.ReactiveNestedSampler(
    ['a_v', 'a_s', 'a_c', 'a_sym', 'sigma_sys'],
    loglike_nopair,
    prior_transform_nopair,
    log_dir='ultranest_nopair'
)
result_nopair = sampler_nopair.run(min_num_live_points=300, dlogz=0.5)

sampler_pairing = ultranest.ReactiveNestedSampler(
    ['a_v', 'a_s', 'a_c', 'a_sym', 'a_p', 'sigma_sys'],
    loglike_with_pairing,
    prior_transform_with_pairing,
    log_dir='ultranest_pairing'
)
result_pairing = sampler_pairing.run(min_num_live_points=300, dlogz=0.5)

print("No-pairing model:")
print("  logZ =", result_nopair['logz'], "±", result_nopair['logzerr'])
print("With-pairing model:")
print("  logZ =", result_pairing['logz'], "±", result_pairing['logzerr'])
print("Bayes factor (pairing / no-pairing) =", np.exp(result_pairing['logz'] - result_nopair['logz']))

Creating directory for new run ultranest_nopair/run2
[ultranest] Sampling 300 live points from prior ...


KeyboardInterrupt: 